# Module 5 — Mô hình LightGBM (Cửa hàng × Nhóm hàng × Ngày)

Dự báo `so_luong` bằng LightGBM trên cùng bảng, cùng 14 đặc trưng và cùng mốc chia với Random Forest. Ba cột `_code` được khai báo là biến phân loại; không thêm thông tin mới vào bảng của Module 3.

Chạy `module3/Module3_1_tien_xu_ly.ipynb` trước nếu chưa có `outputs/daily_store_category.csv`. Sau đó chạy tất cả ô trong notebook này (thư mục làm việc `module5/`), hoặc chạy `python module5/lightgbm_model.py` từ gốc repo.

In [ ]:
import pandas as pd
from lightgbm_model import load_data, tune, train_and_score, analyze, OUT_LGBM, nop_ket_qua
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

## 1. Đặc trưng và chia tập

Dùng `tao_dac_trung`, `chia_tap`, `bang_baseline` của Module 3. LightGBM nhận ba cột mã dưới dạng `categorical_feature`; mọi mô hình vẫn dùng cùng các cột đầu vào. Một quan sát tại ngày t chỉ dùng số bán từ các ngày trước t.

In [ ]:
daily, FEATS, fit, val, tr, te, base_val, rows = load_data()
print('Baseline trên Validation:'); display(base_val)
print('Baseline trên Test:'); display(pd.DataFrame(rows).T)

## 2. Tối ưu tham số trên Validation

Thử 4 hàm mất mát (`regression`, `regression_l1`, `poisson`, `tweedie`) × 3 bộ tham số = 12 cấu hình. Early stopping theo metric của từng hàm mất mát trên Validation, tối đa 1.500 vòng; chọn cấu hình có **MAE Validation** thấp nhất. Lưu bảng thử nghiệm và cấu hình chọn vào `outputs/lightgbm/`.

In [ ]:
tuning, best = tune(FEATS, fit, val)
display(tuning.drop_duplicates('objective').set_index('objective')
        [['num_leaves', 'min_child_samples', 'learning_rate', 'reg_lambda', 'so_vong', 'MAE', 'RMSE', 'WAPE', 'Bias']])

## 3. Train lại trên toàn bộ Train và chấm Test

Train lại trên 15/01–28/02 theo đúng số vòng đã chọn; Test là tháng 3. Dự báo âm được cắt về 0. Lưu mô hình, dự báo và các chỉ số chung.

In [ ]:
model, te, result = train_and_score(FEATS, tr, te, rows, best)
display(result)

## 4. Đặc trưng quan trọng và lỗi theo nhóm hàng / cửa hàng / ngày

Importance của LightGBM là tổng gain trên các nhánh cây; phần trăm trong báo cáo được chuẩn hóa từ giá trị này.

In [ ]:
imp, err_cat, err_store, day = analyze(model, FEATS, te)
display(imp.sort_values(ascending=False).head(10))
display(err_cat.head(10))

## 5. Kết quả và thảo luận

*Số liệu từ lần chạy ngày 10/10/2026 trên bảng mô hình chưa lọc (giữ bán buôn, online, trả lại). Khi thay bộ lọc, cần chạy lại và cập nhật mục này.*

**Tối ưu tham số.** Trong 12 cấu hình, `objective=regression_l1`, `num_leaves=31`, `min_child_samples=100`, `learning_rate=0.03`, `reg_lambda=3`, `n_estimators=153` có MAE Validation thấp nhất: 0,741 so với 0,847 của MA7. Ba cấu hình tốt nhất đều dùng `regression_l1` (MAE 0,741–0,751); Tweedie tốt nhất đạt 0,877.

**Kết quả trên Test (tháng 3, 67.146 ô):**

| Mô hình | MAE | RMSE | WAPE | R² | Bias |
|---|---:|---:|---:|---:|---:|
| Naive (lag 1) | 0,887 | 6,675 | 0,802 | −0,142 | −0,9% |
| Seasonal Naive (lag 7) | 0,908 | 6,728 | 0,821 | −0,160 | +1,6% |
| MA7 | 0,744 | 4,975 | 0,674 | 0,365 | −0,2% |
| **LightGBM** | **0,678** | 5,478 | **0,614** | 0,231 | **−37,3%** |

**Thảo luận.**
1. LightGBM giảm MAE/WAPE 8,9% so với MA7 và thắng WAPE ở 22/23 nhóm hàng, 124/130 cửa hàng. RMSE cao hơn (5,478 so với 4,975) và R² thấp hơn (0,231 so với 0,365).
2. Tổng dự báo chỉ 46.495 so với 74.196 sản phẩm thực tế: trung bình 1.500 so với 2.393 sản phẩm/ngày (MA7 2.388). Trong 278 ô bán ≥ 20 sản phẩm, mô hình dự báo 4.589 so với 13.570 thực tế. Hàm L1 ưu tiên trung vị ở dữ liệu thưa, dẫn tới dự báo thiếu hàng.
3. Mô hình giảm lỗi ở 46.477 ô không bán (MAE 0,079 so với 0,241 của MA7), nhưng kém ở các ô có bán (2,026 so với 1,876). Điện thoại thắng MA7 về WAPE (0,541 so với 0,559); Phụ kiện thua nhẹ (0,513 so với 0,507).
4. Theo gain, quan trọng nhất là `Mã cửa hàng_code` 30,8%, `tb_14` 20,0%, `Nhóm hàng_code` 19,6%, `ngay_trong_thang` 16,2%. Ba cột mã cửa hàng/nhóm/vùng được LightGBM xử lý như biến phân loại.
5. So với Random Forest (MAE 0,757; Bias +0,8%), LightGBM chính xác hơn theo MAE nhưng dự báo thiếu 37,3% tổng lượng. Nếu dùng tiêu chí đề xuất |Bias| ≤ 10% cho bài toán nhập/phân bổ hàng, LightGBM hiện không đạt.

**Giới hạn:** Validation sau Tết khác Test tháng 3; bộ lọc giao dịch và tiêu chí chọn mô hình cần thống nhất trước khi kết luận cuối.

## Nộp kết quả vào báo cáo

Lọc kết quả đã lưu sang `bao_cao/du_lieu/lightgbm/`, rồi từ gốc repo chạy `python module5/tao_bao_cao.py` để tạo lại báo cáo tổng hợp.

In [ ]:
nop_ket_qua('lightgbm', val)